#### Modules import

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
from tqdm import tqdm_notebook as tqdm

#### 自由時報

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

company_list = pd.read_csv("/content/drive/MyDrive/Company.csv").Company
company_list[10:]

In [ ]:
def handleDataframe(search_list):
    df = pd.DataFrame(search_list)
    df = df.explode("Content")
    df = df[df["Content"] != ""]
    df = df[df["Content"] != "請繼續往下閱讀..."]
    df = df[df["Content"] != '\n    一手掌握經濟脈動\n    點我訂閱自由財經Youtube頻道\n']
    df.to_csv('/content/drive/MyDrive/News_liberty_template.csv', mode="a", header=False)
    print("Dataframe saved!")

In [ ]:
for company_name in company_list[10:]:
    keyword = f"{company_name}"
    print(f"Keyword = {keyword}")
    pages = 5
    search_list = []
    for page in range(1, pages + 1):
        # print(f"https://search.ltn.com.tw/list?keyword={keyword}&page={page}")
        res = requests.get(f"https://search.ltn.com.tw/list?keyword={keyword}&page={page}")
        if res.status_code == 200:
            content = res.content
            soup = BeautifulSoup(content, "html.parser")
            # print(content)

            items = soup.findAll("div", {"class": "cont"})
            # print(items)
            for item in items:
                # title
                news_title = item.find("a").get("title")

                # url
                href = item.find("a").get("href")

                # content
                news_content = requests.get(href).content
                news_soup = BeautifulSoup(news_content, "html.parser")
                news = news_soup.find("div", {"class": "text"})
                # news_text = news.text
                news_text = []
                for paragragh in news.findAll("p"):
                    news_text.append(paragragh.text)

                # add item into json object
                search_list.append({
                    "Keyword": keyword,
                    "Name": news_title,
                    # "URL": href,
                    "Content": news_text[1:-1],
                    # "news_from": news_from,
                    # "time_created": time_created
                })
    handleDataframe(search_list)

In [ ]:
company_list[3]

In [ ]:
keyword = f"{company_list[3]}"
pages = 5
search_list = []
for page in range(1, pages + 1):
    # print(f"https://search.ltn.com.tw/list?keyword={keyword}&page={page}")
    res = requests.get(f"https://search.ltn.com.tw/list?keyword={keyword}&page={page}")
    if res.status_code == 200:
        content = res.content
        soup = BeautifulSoup(content, "html.parser")
        # print(content)

        items = soup.findAll("div", {"class": "cont"})
        # print(items)
        for item in items:
            # title
            news_title = item.find("a").get("title")

            # url
            href = item.find("a").get("href")

            # content
            news_content = requests.get(href).content
            news_soup = BeautifulSoup(news_content, "html.parser")
            news = news_soup.find("div", {"class": "text"})
            # news_text = news.text
            news_text = []
            for paragragh in news.findAll("p"):
                news_text.append(paragragh.text)

            # add item into json object
            search_list.append({
                "Keyword": keyword,
                "Name": news_title,
                # "URL": href,
                "Content": news_text[1:-1],
                # "news_from": news_from,
                # "time_created": time_created
            })

In [ ]:
search_list

In [ ]:
df = df_lib[df_lib["news_text"] != ""]
df = df[df["news_text"] != "請繼續往下閱讀..."]
df = df[df["news_text"] != '\n    一手掌握經濟脈動\n    點我訂閱自由財經Youtube頻道\n']
df

In [ ]:
# Save the DataFrame to an Excel file
df.to_csv('/content/drive/MyDrive/News_liberty.csv', index=False)

# Download the file
# from google.colab import files
# files.download('liberty_times_com1.csv')


#### 公司列表

In [ ]:
import requests
from bs4 import BeautifulSoup
import re

res = requests.get(f"https://statementdog.com/taiex/19-semiconductor-industry")
if res.status_code == 200:
    content = res.content
    soup = BeautifulSoup(content, "html.parser")
    # print(content)
    company_names = []
    filter = []
    items = soup.findAll("a", {"class": "industry-stream-company"})
    for item in items:
        name_n_num = item.text
        # company_names.append(name)

        pattern = "(\d+ ).+"
        match_num = re.match(pattern, name_n_num)
        name = name_n_num.replace(match_num.group(1), "")

        company_names.append(name)


In [ ]:
company_names

In [ ]:
import pandas as pd

df_com = pd.DataFrame(set(company_names))
df_com.columns = ['Company']
df_com.reset_index()
df_com

In [ ]:
# Save the DataFrame to an Excel file
df_com.to_csv('Company.csv')

# Download the file
from google.colab import files
files.download('Company.csv')

#### 工商時報

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service

driver_path = '/content/drive/MyDrive/chromedriver.exe'
service = Service(executable_path=driver_path)
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service=service, options=options)

In [ ]:
# 操作 browser 的 API
from selenium.webdriver.chrome.service import Service
from selenium import webdriver

# 處理逾時例外的工具
from selenium.common.exceptions import TimeoutException

# 面對動態網頁，等待某個元素出現的工具，通常與 exptected_conditions 搭配
from selenium.webdriver.support.ui import WebDriverWait

# 搭配 WebDriverWait 使用，對元素狀態的一種期待條件，若條件發生，則等待結束，往下一行執行
from selenium.webdriver.support import expected_conditions as EC

# 期待元素出現要透過什麼方式指定，通常與 EC、WebDriverWait 一起使用
from selenium.webdriver.common.by import By

# 強制等待 (執行期間休息一下)
from time import sleep

# 整理 json 使用的工具
import json

# 執行 command 的時候用的
import os

# 啟動瀏覽器工具的選項
options = webdriver.ChromeOptions()
# options.add_argument("--headless")              #不開啟實體瀏覽器背景執行
options.add_argument("--start-maximized")         #最大化視窗
options.add_argument("--incognito")               #開啟無痕模式
options.add_argument("--disable-popup-blocking") #禁用彈出攔截

# 使用 Chrome 的 WebDriver
service = Service(executable_path="./chromedriver.exe")
driver = webdriver.Chrome(
    options = options,
    service = service
)

In [ ]:
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service

# Define Chrome options
options = webdriver.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
driver = webdriver.Chrome(service=service, options=options)


In [ ]:
res1 = requests.get("https://www.ctee.com.tw/search/ESG")

if res1.status_code == 200:
    content = res1.content
    soup = BeautifulSoup(content, "html.parser")
    # print(content)
    search_list1 = []
    items = soup.findAll("div", {"class": "newslist__card"})

    for item in items:
        # title
        news_title = item.find("h3", {"class": "news-title"}).find("a").text

        # url
        href = "https://www.ctee.com.tw" + item.find("h3", {"class": "news-title"}).find("a").get("href")

        # content
        news_content = requests.get(href).content
        news_soup = BeautifulSoup(news_content, "html.parser")
        news = news_soup.find("article")
        # news_text = news.text
        news_text = []
        for paragragh in news.findAll("p"):
            news_text.append(paragragh.text)

        # add item into json object
        search_list1.append({
            "news_title": news_title,
            "news_link": href,
            "news_text": news_text[1:],
            # "news_from": news_from,
            # "time_created": time_created
        })

In [ ]:
search_list1

In [ ]:
df1 = pd.DataFrame(search_list1)
df1 = df1.explode("news_text")
df1